# FDA Purple Book — Biological Approvals Analysis 2020–2026

**Data source:** FDA Purple Book monthly historical data CSVs  
**URL pattern:** `https://www.accessdata.fda.gov/drugsatfda_docs/PurpleBook/{YEAR}/purplebook-search-{MONTH}-data-download.csv`  
**Coverage:** 77 monthly files, February 2020 – April 2026  
**Scope:** Biological License Applications (BLAs) only — 351(a) reference biologics, 351(k) biosimilars, and interchangeables

## 1. Download Monthly CSVs

In [ ]:
import os
import urllib.request

OUT_DIR = r'E:\Work\AI_Drug\purplebook_csvs'
os.makedirs(OUT_DIR, exist_ok=True)

MONTHS = [
    'january','february','march','april','may','june',
    'july','august','september','october','november','december'
]
BASE_URL = 'https://www.accessdata.fda.gov/drugsatfda_docs/PurpleBook'

results = []
for year in range(2020, 2027):
    max_month = 5 if year == 2026 else 12
    for i in range(1, max_month + 1):
        month = MONTHS[i - 1]
        url = f'{BASE_URL}/{year}/purplebook-search-{month}-data-download.csv'
        out_file = os.path.join(OUT_DIR, f'{year}_{month}.csv')
        try:
            urllib.request.urlretrieve(url, out_file)
            size = os.path.getsize(out_file)
            results.append((f'{year}_{month}', 'OK', size))
        except Exception as e:
            results.append((f'{year}_{month}', f'FAILED: {e}', 0))

ok = sum(1 for _, s, _ in results if s == 'OK')
failed = [(n, s) for n, s, _ in results if s != 'OK']
print(f'Downloaded: {ok}  |  Failed: {len(failed)}')
for name, status in failed:
    print(f'  {name}: {status}')

## 2. Load & Deduplicate

In [ ]:
import csv
import glob
from datetime import datetime
from collections import defaultdict

DATA_DIR = r'E:\Work\AI_Drug\purplebook_csvs'

# Key = (BLA Number, Product Number, Supplement Number) — dedup across monthly snapshots
all_new = {}

for f in sorted(glob.glob(os.path.join(DATA_DIR, '*.csv'))):
    fname = os.path.basename(f)
    # 2020_february.csv is a full DB dump with no N/R/U change flags — skip
    if fname == '2020_february.csv':
        continue
    with open(f, encoding='utf-8-sig', errors='replace') as fh:
        lines = fh.readlines()
    # Find the header row (starts with 'N/R/U')
    hdr_idx = next(
        (i for i, line in enumerate(lines) if line.strip().lstrip(',').startswith('N/R/U')),
        None
    )
    if hdr_idx is None:
        continue
    for row in csv.DictReader(lines[hdr_idx:]):
        if row.get('N/R/U', '').strip() != 'N':
            continue
        key = (row.get('BLA Number','').strip(),
               row.get('Product Number','').strip(),
               row.get('Supplement Number','').strip())
        if key not in all_new:
            all_new[key] = row

print(f'Unique newly-approved (N flag) entries across all files: {len(all_new)}')

## 3. Parse Dates & Filter 2020–2026

In [ ]:
DATE_FMTS = [
    '%d-%b-%y', '%d-%b-%Y',   # 14-Dec-12, 14-Dec-2012
    '%m/%d/%Y', '%m/%d/%y',   # 12/31/2002, 9/27/22
    '%Y-%m-%d',               # 2024-03-13
    '%B %d, %Y',              # March 13, 2024
]

def parse_dt(s):
    s = s.strip()
    for fmt in DATE_FMTS:
        try:
            return datetime.strptime(s, fmt)
        except ValueError:
            pass
    return None

approved = []
unparseable = []

for key, row in all_new.items():
    dt = parse_dt(row.get('Approval Date', ''))
    if dt is None:
        unparseable.append(row.get('Approval Date', ''))
        continue
    if 2020 <= dt.year <= 2026:
        approved.append({'year': dt.year, '_dt': dt, **row})

print(f'Entries approved 2020–2026: {len(approved)}')
print(f'Unparseable dates:          {len(unparseable)}')
if unparseable:
    print('  Samples:', list(set(unparseable))[:5])

## 4. Approvals by Year

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(approved)

by_year = df.groupby('year').size().reset_index(name='count')
print(by_year.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(by_year['year'], by_year['count'], color='steelblue', edgecolor='white')
ax.set_xlabel('Year')
ax.set_ylabel('Approvals')
ax.set_title('FDA Purple Book — New Biological Approvals by Year (2020–2026)')
ax.set_xticks(by_year['year'])
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            int(bar.get_height()), ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

## 5. BLA Type Breakdown (Reference vs Biosimilar vs Interchangeable)

In [ ]:
def classify_type(t):
    t = str(t).strip()
    if 'Interchangeable' in t:
        return '351(k) Interchangeable'
    if 'Biosimilar' in t:
        return '351(k) Biosimilar'
    if '351(a)' in t:
        return '351(a) Reference'
    return 'Other/Unknown'

df['bla_class'] = df['BLA Type'].apply(classify_type)

# Year × type pivot
pivot = df.pivot_table(index='year', columns='bla_class', aggfunc='size', fill_value=0)
print(pivot)

pivot.plot(kind='bar', stacked=True, figsize=(10, 5),
           color=['#2196F3','#4CAF50','#FF9800','#9E9E9E'])
plt.title('Approvals by BLA Type per Year')
plt.xlabel('Year')
plt.ylabel('Count')
plt.legend(title='BLA Type', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 6. FDA Center (CBER vs CDER)

In [ ]:
center = df['Center'].str.strip().value_counts()
print(center)

center.plot(kind='pie', autopct='%1.1f%%', figsize=(5, 5),
            colors=['#1565C0','#C62828'], startangle=140)
plt.title('Approvals by FDA Center (2020–2026)')
plt.ylabel('')
plt.tight_layout()
plt.show()

## 7. Top Applicants

In [ ]:
top_apps = df['Applicant'].str.strip().value_counts().head(20)
print(top_apps)

fig, ax = plt.subplots(figsize=(10, 6))
top_apps[::-1].plot(kind='barh', ax=ax, color='teal')
ax.set_xlabel('Number of Approvals')
ax.set_title('Top 20 Applicants — Purple Book Approvals 2020–2026')
plt.tight_layout()
plt.show()

## 8. Biosimilar Competition — Top Reference Products

In [ ]:
biosim = df[df['bla_class'].isin(['351(k) Biosimilar', '351(k) Interchangeable'])].copy()
print(f'Total biosimilar/interchangeable entries: {len(biosim)}')

ref_col = 'Ref. Product Proper Name'
ref_counts = (
    biosim[ref_col].str.strip()
    .replace({'N/A': None, '-': None, '': None})
    .dropna()
    .value_counts()
    .head(15)
)
print(ref_counts)

fig, ax = plt.subplots(figsize=(10, 6))
ref_counts[::-1].plot(kind='barh', ax=ax, color='darkorange')
ax.set_xlabel('Number of Biosimilar Entries Approved')
ax.set_title('Top Reference Products by Biosimilar Competition (2020–2026)')
plt.tight_layout()
plt.show()

## 9. Route of Administration & Dosage Form

In [ ]:
route = df['Route of Administration'].str.strip().value_counts().head(12)
form  = df['Dosage Form'].str.strip().value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

route[::-1].plot(kind='barh', ax=axes[0], color='#5C6BC0')
axes[0].set_title('Route of Administration')
axes[0].set_xlabel('Count')

form[::-1].plot(kind='barh', ax=axes[1], color='#26A69A')
axes[1].set_title('Dosage Form')
axes[1].set_xlabel('Count')

plt.suptitle('Route & Dosage Form — Purple Book 2020–2026', fontsize=13)
plt.tight_layout()
plt.show()

## 10. Orphan Drug Approvals by Year

In [ ]:
orphan_col = 'Orphan Exclusivity Exp. Date'
orphan = df[~df[orphan_col].str.strip().isin(['', 'N/A', '-'])]
print(f'Products with orphan exclusivity: {len(orphan)}')
print(orphan.groupby('year').size())

orphan_by_year = orphan.groupby('year').size()
orphan_by_year.plot(kind='bar', figsize=(8, 4), color='purple', edgecolor='white')
plt.title('Orphan Drug Approvals by Year (Purple Book 2020–2026)')
plt.xlabel('Year')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 11. Recent Original Approvals (2025–2026 Spotlight)

In [ ]:
originals = df[df['Submission Type'].str.strip().isin(['Original', 'Original Application'])].copy()
recent = (
    originals[originals['year'] >= 2025]
    .sort_values('_dt', ascending=False)
    [['Approval Date', 'Proprietary Name', 'Proper Name', 'BLA Type', 'Applicant', 'Center']]
    .drop_duplicates(subset=['Proprietary Name', 'Proper Name', 'Approval Date'])
)
print(recent.to_string(index=False))

## 12. Export Summary Table

In [ ]:
# Save full approved dataset to Excel for further review
out_cols = [
    'year', 'Approval Date', 'Proprietary Name', 'Proper Name',
    'BLA Type', 'bla_class', 'Applicant', 'Center', 'Submission Type',
    'Route of Administration', 'Dosage Form', 'BLA Number',
    'Ref. Product Proper Name', 'Orphan Exclusivity Exp. Date'
]
export = df[[c for c in out_cols if c in df.columns]]
export_path = r'E:\Work\AI_Drug\purplebook_2020_2026_approved.xlsx'
export.to_excel(export_path, index=False)
print(f'Exported {len(export)} rows → {export_path}')

## 13. Merge Proper Name Variants

In [ ]:
def normalize_merged_name(name, original_names, hyphen_base_counts):
    if pd.isna(name):
        return name
    name = str(name).strip()
    if not name:
        return name
    if '-' in name:
        base = name.split('-', 1)[0].strip()
        if base and (base in original_names or hyphen_base_counts.get(base, 0) > 1):
            return base
    return name

df_merged = df.copy()
df_merged['original proper name'] = df_merged['Proper Name'].astype(str).str.strip()

original_names = set(df_merged['original proper name'].dropna().unique())
hyphen_base_counts = (
    df_merged.loc[df_merged['original proper name'].str.contains('-', na=False), 'original proper name']
    .str.split('-', 1).str[0].str.strip()
    .value_counts()
)

df_merged['Proper Name, merged'] = df_merged['original proper name'].apply(
    lambda n: normalize_merged_name(n, original_names, hyphen_base_counts)
)

print('Original unique proper names:', df_merged['original proper name'].nunique())
print('Merged unique proper names:  ', df_merged['Proper Name, merged'].nunique())
print(df_merged['Proper Name, merged'].value_counts().head(25).to_string())

def join_unique(series):
    values = series.dropna().astype(str).str.strip()
    values = [v for v in dict.fromkeys(values) if v and v not in ['nan', 'None']]
    return '|'.join(values)

group_cols = [
    c for c in df_merged.columns
    if c not in ['_dt', 'Proper Name', 'Proper Name, merged', 'original proper name']
]

merged = (
    df_merged.groupby('Proper Name, merged', sort=False)[group_cols]
    .agg(join_unique)
    .reset_index()
)

merged['original proper name'] = (
    df_merged.groupby('Proper Name, merged', sort=False)['original proper name']
    .agg(join_unique)
    .values
)

cols = ['Proper Name, merged', 'original proper name'] + [
    c for c in merged.columns if c not in ['Proper Name, merged', 'original proper name']
]
merged = merged[cols]

print(f'After merge: {len(merged)} grouped rows')
display(merged.loc[merged['Proper Name, merged'].isin(['ranibizumab', 'ravulizumab'])].head(10))

## 13. Merge Proper Name Variants


In [ ]:
def normalize_merged_name(name, original_names, hyphen_base_counts):
    if pd.isna(name):
        return name
    name = str(name).strip()
    if not name:
        return name
    if '-' in name:
        base = name.split('-', 1)[0].strip()
        if base and (base in original_names or hyphen_base_counts.get(base, 0) > 1):
            return base
    return name

df_merged = df.copy()
df_merged['original proper name'] = df_merged['Proper Name'].astype(str).str.strip()

original_names = set(df_merged['original proper name'].dropna().unique())
hyphen_base_counts = (
    df_merged.loc[df_merged['original proper name'].str.contains('-', na=False), 'original proper name']
    .str.split('-', 1).str[0].str.strip()
    .value_counts()
)

df_merged['Proper Name, merged'] = df_merged['original proper name'].apply(
    lambda n: normalize_merged_name(n, original_names, hyphen_base_counts)
)

print('Original unique proper names:', df_merged['original proper name'].nunique())
print('Merged unique proper names:  ', df_merged['Proper Name, merged'].nunique())
print(df_merged['Proper Name, merged'].value_counts().head(25).to_string())

def join_unique(series):
    values = series.dropna().astype(str).str.strip()
    values = [v for v in dict.fromkeys(values) if v and v not in ['nan', 'None']]
    return '|'.join(values)

group_cols = [
    c for c in df_merged.columns
    if c not in ['_dt', 'Proper Name', 'Proper Name, merged', 'original proper name']
]

merged = (
    df_merged.groupby('Proper Name, merged', sort=False)[group_cols]
    .agg(join_unique)
    .reset_index()
)

merged['original proper name'] = (
    df_merged.groupby('Proper Name, merged', sort=False)['original proper name']
    .agg(join_unique)
    .values
)

cols = ['Proper Name, merged', 'original proper name'] + [
    c for c in merged.columns if c not in ['Proper Name, merged', 'original proper name']
]
merged = merged[cols]

print(f'After merge: {len(merged)} grouped rows')
display(merged.loc[merged['Proper Name, merged'].isin(['ranibizumab', 'ravulizumab'])].head(10))
